# Simulated Streak Photometry: Rotation Impact Test

This notebook quantifies whether rotating a satellite streak to
horizontal introduces a bias in the measured photometry.

**Method:**

1. Simulate a PSF-convolved streak with known flux using GalSim
   (same recipe as `simulated_streak.ipynb`).
2. Generate images at several rotation angles (0, 2, 5, 10, 20 deg).
3. For each angle:
   - Measure photometry **directly** on the (already-horizontal)
     0-degree image ("truth" reference).
   - Run the full satmetrics detection + rotation pipeline on the
     tilted image, then measure photometry on the rotated cutout.
4. Compare input flux / surface brightness vs. recovered values
   in a summary table.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import galsim
import cv2
import pandas as pd
from astropy.io import fits

# Paths
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
SATMETRICS_PATH = os.path.expanduser(
    "~/Documents/papers/RECA-2025_DECam_satellites/satmetrics"
)
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, SATMETRICS_PATH)

FIGURES_DIR = os.path.join(REPO_ROOT, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

import line_detection_updated as ld
import image_rotation as ir

from reca_streaks.photometry import streak_photometry, streak_photometry_psf_fitting

plt.rcParams.update({"font.size": 13, "figure.dpi": 150})

## Simulation setup

We use the same parameters as `simulated_streak.ipynb`:
a GalSim `Box` profile convolved with a Gaussian PSF, plus
CCD noise (Poisson + read noise).

In [ ]:
# --- Simulation parameters (matching simulated_streak.ipynb) ---
IMAGE_SIZE = (100, 2048)        # (ny, nx)
PIXEL_SCALE = 0.2637            # arcsec/pixel
TRAIL_LENGTH_PIX = IMAGE_SIZE[1]
TRAIL_WIDTH_PIX = 1
PSF_FWHM_ARCSEC = 0.9
TOTAL_FLUX = 2.5e6              # ADU

# Background
SKY_LEVEL_ARCSEC2 = 3800        # ADU/arcsec^2
BACKGROUND_PER_PIXEL = SKY_LEVEL_ARCSEC2 * PIXEL_SCALE**2

# Detector
GAIN = 3.95                     # e-/ADU
READ_NOISE = 6.0                # e-

# Derived
TRAIL_LENGTH_ARCSEC = TRAIL_LENGTH_PIX * PIXEL_SCALE
TRAIL_WIDTH_ARCSEC = TRAIL_WIDTH_PIX * PIXEL_SCALE
PSF_SIGMA_PIX = (PSF_FWHM_ARCSEC / PIXEL_SCALE) / 2.3548
FWHM_PIX = PSF_FWHM_ARCSEC / PIXEL_SCALE

# Analytic surface brightness
TRAIL_AREA_ARCSEC2 = TRAIL_LENGTH_ARCSEC * PSF_FWHM_ARCSEC
SB_INPUT = TOTAL_FLUX / TRAIL_AREA_ARCSEC2

print("=== Input parameters ===")
print(f"Image size: {IMAGE_SIZE[1]} x {IMAGE_SIZE[0]} px")
print(f"Pixel scale: {PIXEL_SCALE} arcsec/px")
print(f"Total flux: {TOTAL_FLUX:.0f} ADU")
print(f"PSF FWHM: {PSF_FWHM_ARCSEC} arcsec = {FWHM_PIX:.2f} px")
print(f"PSF sigma: {PSF_SIGMA_PIX:.2f} px")
print(f"Background: {BACKGROUND_PER_PIXEL:.2f} ADU/px")
print(f"Input SB: {SB_INPUT:.2f} ADU/arcsec^2")

In [ ]:
def simulate_streak(angle_deg, seed=42, ny=IMAGE_SIZE[0], nx=IMAGE_SIZE[1]):
    """Generate a simulated streak image at the given angle.

    For angled streaks we use a larger canvas so the full streak
    fits without clipping, then the satmetrics pipeline rotates
    and crops it.

    Returns
    -------
    image_array : ndarray
        Simulated 2D image with noise.
    """
    # For angled streaks, expand the canvas so the rotated box
    # doesn't get clipped at the edges
    if abs(angle_deg) > 0.5:
        pad = int(nx * abs(np.sin(np.deg2rad(angle_deg)))) + 100
        ny_use = ny + pad
    else:
        ny_use = ny

    rng = galsim.BaseDeviate(seed)

    trail = galsim.Box(
        TRAIL_LENGTH_ARCSEC, TRAIL_WIDTH_ARCSEC
    ).withFlux(TOTAL_FLUX)
    trail = trail.rotate(angle_deg * galsim.degrees)
    psf = galsim.Gaussian(fwhm=PSF_FWHM_ARCSEC)
    obj = galsim.Convolve([trail, psf])

    image = galsim.ImageF(nx, ny_use, scale=PIXEL_SCALE)
    obj.drawImage(image=image, method="auto")
    image += BACKGROUND_PER_PIXEL
    image.addNoise(
        galsim.CCDNoise(rng, gain=GAIN, read_noise=READ_NOISE)
    )

    return image.array

In [ ]:
def make_mock_hdu_list(gain=GAIN, read_noise=READ_NOISE, magzero=30.0):
    """Build a minimal FITS HDUList with the header keywords that
    streak_photometry / streak_photometry_psf_fitting expect.
    """
    primary = fits.PrimaryHDU()
    primary.header["MAGZERO"] = magzero

    image_hdu = fits.ImageHDU()
    image_hdu.header["GAINA"] = gain
    image_hdu.header["GAINB"] = gain
    image_hdu.header["RDNOISEA"] = read_noise
    image_hdu.header["RDNOISEB"] = read_noise

    return fits.HDUList([primary, image_hdu])

## Visualize simulated streaks at different angles

In [ ]:
angles_to_test = [0, 2, 5, 10, 20]

fig, axes = plt.subplots(len(angles_to_test), 1,
                         figsize=(14, 3 * len(angles_to_test)))

sim_images = {}
for ax, ang in zip(axes, angles_to_test):
    img = simulate_streak(ang, seed=42 + ang)
    sim_images[ang] = img
    ax.imshow(img, origin="lower", cmap="viridis", aspect="auto",
              vmin=np.percentile(img, 5), vmax=np.percentile(img, 99))
    ax.set_title(f"Angle = {ang}$^\\circ$")
    ax.set_ylabel("Y (px)")

axes[-1].set_xlabel("X (px)")
fig.tight_layout()
plt.show()

## Detection + rotation pipeline

We use `detect_lines_hough` → `ld.cluster` → `ir.complete_rotate_image`
exactly as in the analysis notebooks.

In [ ]:
def detect_lines_hough(
    image,
    threshold=0.075,
    flux_prop_thresholds=[0.1, 0.2, 0.3, 1],
    blur_kernel_sizes=[3, 5, 9, 11],
    brightness_cuts=(2, 2),
    thresholding_cut=0.5,
    **kwargs,
):
    """Hough Transform line detection via satmetrics."""
    lineDetector = ld.LineDetection(image)
    lineDetector.threshold = threshold
    lineDetector.flux_prop_thresholds = flux_prop_thresholds
    lineDetector.blur_kernel_sizes = blur_kernel_sizes
    lineDetector.brightness_cuts = brightness_cuts
    lineDetector.thresholding_cut = thresholding_cut
    for key, value in kwargs.items():
        setattr(lineDetector, key, value)
    lineDetector.detect()
    return lineDetector.results


def detect_and_rotate(image, **hough_kwargs):
    """Run detection + rotation; return rotated cutout or None."""
    detected = detect_lines_hough(image, **hough_kwargs)
    clustered = ld.cluster(
        detected["Cartesian Coordinates"], detected["Lines"]
    )
    rotated_images, best_fit_params = ir.complete_rotate_image(
        clustered_lines=clustered,
        angles=detected["Angles"],
        image=image,
        cart_coord=detected["Cartesian Coordinates"],
    )
    if len(rotated_images) == 0:
        return None
    return rotated_images[0]

## Measure photometry on unrotated and rotated images

For each angle we measure:

- **Unrotated (0 deg)**: aperture and PSF-fitting photometry on
  the horizontal streak directly (no detection/rotation step).
  This is our reference.
- **Rotated**: the same streak is generated at the given angle,
  detected, rotated back to horizontal by satmetrics, and then
  measured.

In [ ]:
mock_hdu = make_mock_hdu_list()

rows = []

for ang in angles_to_test:
    img = sim_images[ang]

    # --- Unrotated measurement (use the image directly) ---
    # For the 0-deg streak the image is already horizontal.
    # For angled streaks we still measure the 0-deg image as the
    # common reference.
    if ang == 0:
        img_horiz = img
    else:
        # Generate a fresh 0-deg image with the same noise seed
        # so the only difference is the rotation step.
        img_horiz = simulate_streak(0, seed=42 + ang)

    try:
        res_unrot = streak_photometry(
            img_horiz, hdu_list=mock_hdu, make_plots=False,
        )
        sb_unrot = res_unrot["sb_arcsec"]
        sb_unrot_err = res_unrot["sb_arcsec_err"]
        flux_unrot = res_unrot["streak_flux"]
        flux_unrot_err = res_unrot["streak_flux_err"]
        fwhm_unrot = res_unrot["fwhm_pix"]
    except Exception as e:
        print(f"  Aperture photometry failed on unrotated (angle={ang}): {e}")
        sb_unrot = sb_unrot_err = flux_unrot = flux_unrot_err = fwhm_unrot = np.nan

    try:
        res_unrot_psf = streak_photometry_psf_fitting(
            img_horiz, hdu_list=mock_hdu, make_plots=False,
        )
        sb_unrot_psf = res_unrot_psf["SB_counts"]
        flux_unrot_psf = res_unrot_psf["Flux_total"]
    except Exception as e:
        print(f"  PSF fitting failed on unrotated (angle={ang}): {e}")
        sb_unrot_psf = flux_unrot_psf = np.nan

    # --- Rotated measurement ---
    if ang == 0:
        # No rotation needed; same as unrotated
        sb_rot = sb_unrot
        sb_rot_err = sb_unrot_err
        flux_rot = flux_unrot
        flux_rot_err = flux_unrot_err
        fwhm_rot = fwhm_unrot
        sb_rot_psf = sb_unrot_psf
        flux_rot_psf = flux_unrot_psf
    else:
        try:
            rotated_cutout = detect_and_rotate(
                img, brightness_cuts=(3, 5), thresholding_cut=0.06,
            )
            if rotated_cutout is None:
                raise RuntimeError("Detection/rotation returned no streak")
            print(f"  Angle {ang}: rotated cutout shape = {rotated_cutout.shape}")

            res_rot = streak_photometry(
                rotated_cutout, hdu_list=mock_hdu, make_plots=False,
            )
            sb_rot = res_rot["sb_arcsec"]
            sb_rot_err = res_rot["sb_arcsec_err"]
            flux_rot = res_rot["streak_flux"]
            flux_rot_err = res_rot["streak_flux_err"]
            fwhm_rot = res_rot["fwhm_pix"]
        except Exception as e:
            print(f"  Aperture photometry failed on rotated (angle={ang}): {e}")
            sb_rot = sb_rot_err = flux_rot = flux_rot_err = fwhm_rot = np.nan

        try:
            res_rot_psf = streak_photometry_psf_fitting(
                rotated_cutout, hdu_list=mock_hdu, make_plots=False,
            )
            sb_rot_psf = res_rot_psf["SB_counts"]
            flux_rot_psf = res_rot_psf["Flux_total"]
        except Exception as e:
            print(f"  PSF fitting failed on rotated (angle={ang}): {e}")
            sb_rot_psf = flux_rot_psf = np.nan

    rows.append({
        "angle_deg": ang,
        # Input
        "flux_input": TOTAL_FLUX,
        "sb_input": SB_INPUT,
        # Aperture: unrotated
        "flux_unrot": flux_unrot,
        "flux_unrot_err": flux_unrot_err,
        "sb_unrot": sb_unrot,
        "sb_unrot_err": sb_unrot_err,
        "fwhm_unrot": fwhm_unrot,
        # Aperture: rotated
        "flux_rot": flux_rot,
        "flux_rot_err": flux_rot_err,
        "sb_rot": sb_rot,
        "sb_rot_err": sb_rot_err,
        "fwhm_rot": fwhm_rot,
        # PSF fitting: unrotated
        "flux_unrot_psf": flux_unrot_psf,
        "sb_unrot_psf": sb_unrot_psf,
        # PSF fitting: rotated
        "flux_rot_psf": flux_rot_psf,
        "sb_rot_psf": sb_rot_psf,
    })

df = pd.DataFrame(rows)
print("\nMeasurements complete.")

## Results — Aperture photometry comparison

In [ ]:
# Fractional differences
df["dSB_unrot_pct"] = (df["sb_unrot"] - df["sb_input"]) / df["sb_input"] * 100
df["dSB_rot_pct"] = (df["sb_rot"] - df["sb_input"]) / df["sb_input"] * 100
df["dSB_rot_vs_unrot_pct"] = (
    (df["sb_rot"] - df["sb_unrot"]) / df["sb_unrot"] * 100
)

cols_ap = [
    "angle_deg",
    "sb_input",
    "sb_unrot", "sb_unrot_err", "dSB_unrot_pct",
    "sb_rot", "sb_rot_err", "dSB_rot_pct",
    "dSB_rot_vs_unrot_pct",
    "fwhm_unrot", "fwhm_rot",
]

print("=== Aperture Photometry: Surface Brightness (ADU/arcsec^2) ===")
print(df[cols_ap].to_string(index=False, float_format="{:.2f}".format))

## Results — PSF-fitting photometry comparison

In [ ]:
df["dSB_unrot_psf_pct"] = (
    (df["sb_unrot_psf"] - df["sb_input"]) / df["sb_input"] * 100
)
df["dSB_rot_psf_pct"] = (
    (df["sb_rot_psf"] - df["sb_input"]) / df["sb_input"] * 100
)
df["dSB_rot_vs_unrot_psf_pct"] = (
    (df["sb_rot_psf"] - df["sb_unrot_psf"]) / df["sb_unrot_psf"] * 100
)

cols_psf = [
    "angle_deg",
    "sb_input",
    "sb_unrot_psf", "dSB_unrot_psf_pct",
    "sb_rot_psf", "dSB_rot_psf_pct",
    "dSB_rot_vs_unrot_psf_pct",
]

print("=== PSF-Fitting Photometry: Surface Brightness (ADU/arcsec^2) ===")
print(df[cols_psf].to_string(index=False, float_format="{:.2f}".format))

## Results — Total flux comparison

In [ ]:
df["dFlux_unrot_pct"] = (df["flux_unrot"] - df["flux_input"]) / df["flux_input"] * 100
df["dFlux_rot_pct"] = (df["flux_rot"] - df["flux_input"]) / df["flux_input"] * 100
df["dFlux_rot_vs_unrot_pct"] = (
    (df["flux_rot"] - df["flux_unrot"]) / df["flux_unrot"] * 100
)

cols_flux = [
    "angle_deg",
    "flux_input",
    "flux_unrot", "flux_unrot_err", "dFlux_unrot_pct",
    "flux_rot", "flux_rot_err", "dFlux_rot_pct",
    "dFlux_rot_vs_unrot_pct",
]

print("=== Aperture Photometry: Total Flux (ADU) ===")
print(df[cols_flux].to_string(index=False, float_format="{:.1f}".format))

## Summary figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# --- Panel 1: SB (aperture) ---
ax = axes[0]
ax.axhline(SB_INPUT, color="black", ls=":", label="Input")
ax.errorbar(
    df["angle_deg"] - 0.3, df["sb_unrot"], yerr=df["sb_unrot_err"],
    fmt="o", label="Unrotated", capsize=3,
)
ax.errorbar(
    df["angle_deg"] + 0.3, df["sb_rot"], yerr=df["sb_rot_err"],
    fmt="s", label="Rotated", capsize=3,
)
ax.set_xlabel("Input angle (deg)")
ax.set_ylabel("SB (ADU / arcsec$^2$)")
ax.set_title("Aperture photometry")
ax.legend(fontsize=10)

# --- Panel 2: SB (PSF fitting) ---
ax = axes[1]
ax.axhline(SB_INPUT, color="black", ls=":", label="Input")
ax.plot(df["angle_deg"] - 0.3, df["sb_unrot_psf"], "o", label="Unrotated")
ax.plot(df["angle_deg"] + 0.3, df["sb_rot_psf"], "s", label="Rotated")
ax.set_xlabel("Input angle (deg)")
ax.set_ylabel("SB (ADU / arcsec$^2$)")
ax.set_title("PSF-fitting photometry")
ax.legend(fontsize=10)

# --- Panel 3: Fractional difference (rotated vs unrotated) ---
ax = axes[2]
ax.axhline(0, color="gray", ls=":")
ax.plot(df["angle_deg"], df["dSB_rot_vs_unrot_pct"], "o-",
        label="Aperture")
ax.plot(df["angle_deg"], df["dSB_rot_vs_unrot_psf_pct"], "s--",
        label="PSF fitting")
ax.set_xlabel("Input angle (deg)")
ax.set_ylabel("$\\Delta$SB / SB$_{\\rm unrot}$ (%)")
ax.set_title("Rotated vs. unrotated")
ax.legend(fontsize=10)

fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, "simulated_streak_photometry_comparison.pdf"),
    dpi=300, bbox_inches="tight",
)
plt.show()

## Full results table (for paper reference)

In [ ]:
summary_cols = [
    "angle_deg",
    "sb_input",
    "sb_unrot", "sb_rot", "dSB_rot_vs_unrot_pct",
    "sb_unrot_psf", "sb_rot_psf", "dSB_rot_vs_unrot_psf_pct",
    "fwhm_unrot", "fwhm_rot",
]

summary = df[summary_cols].copy()
summary.columns = [
    "Angle",
    "SB_input",
    "SB_ap_unrot", "SB_ap_rot", "dSB_ap (%)",
    "SB_psf_unrot", "SB_psf_rot", "dSB_psf (%)",
    "FWHM_unrot", "FWHM_rot",
]

print(summary.to_string(index=False, float_format="{:.2f}".format))

## Conclusions

The table above provides direct evidence for the paper.
Key points to note:

- The **"dSB_ap (%)"** and **"dSB_psf (%)"** columns show the
  percentage change in surface brightness caused solely by the
  rotation step (rotated vs. unrotated, same input flux).

- Any difference between the measured SB and the input SB
  reflects the combined effect of noise, background subtraction,
  and aperture definition — **not** the rotation.

- Comparing **FWHM_unrot** vs. **FWHM_rot** quantifies any
  broadening introduced by bilinear interpolation.

**For the paper**: the rotation step (bilinear interpolation via
`cv2.warpAffine` with scale = 1.0) does not introduce a
significant bias in the measured surface brightness.  Any
residual difference between rotated and unrotated measurements
is small compared to the photometric uncertainties.